In [1]:
import glob
import os

import numpy as np
import pandas as pd
from cogent3 import get_app
from cogent3.maths.measure import jsd

load_json_app = get_app("load_json")

In [2]:
from clock_project.plot_utils.src.theoretical_analysis import (
    get_ens_abs_diff,
    get_ingroup_jsd,
    get_jad_difference,
    get_nabla_abs_diff,
    plot_time_grouped_scatter_2x2,
)

ModuleNotFoundError: No module named 'clock_project.plot_utils'

In [ ]:
# relative path
triple_model_ditting_dir = (
    "/Users/gulugulu/clock/mammal_orthologs_hsap_1/triples_model_fitting"
)
gene_paths = glob.glob(os.path.join(triple_model_ditting_dir, "*/"))

## Variables - Evolutionary time, Substitution matrix and Initial nucleotide distirbution

In [ ]:
import json

valid_matrix_full_path = (
    "/Users/gulugulu/clock/mammal_orthologs_hsap_1/valid_matrix_full.json"
)
valid_matrix_full = json.load(open(valid_matrix_full_path, "r"))
matrices_list = []
for gene, matrices in valid_matrix_full.items():
    matrices_list.extend(matrices)

Create dataframe

In [ ]:
t_range = [0.5, 1, 1.5, 2]
pi0 = [0.25, 0.25, 0.25, 0.25]
# 0.45
pi4 = [0.06, 0.47, 0.08, 0.39]


def compute_aginst_Q(matrices_list, pi, t_range):

    result_list = []
    for i in range(len(matrices_list)):
        q_pair = list(matrices_list[i].values())
        Q1 = np.array(q_pair[0])
        Q2 = np.array(q_pair[1])

        for t in t_range:
            jsd_value = get_ingroup_jsd(pi, Q1, Q2, t)
            jsd_diff = get_jad_difference(pi, Q1, Q2, t)
            ens_abs_diff = get_ens_abs_diff(pi, Q1, Q2, t)
            nabla_abs_diff = get_nabla_abs_diff(pi, Q1, Q2, t)
            result_list.append(
                (i, t, ens_abs_diff, nabla_abs_diff, jsd_value, jsd_diff)
            )

    df = pd.DataFrame(
        result_list,
        columns=[
            "Matrix_ID",
            "Time",
            "ENS_abs_difference",
            "Nabla_abs_diff",
            "Ingroup_JSD",
            "JSD_difference",
        ],
    )
    return df


df_low = compute_aginst_Q(matrices_list, pi0, t_range)
df_high = compute_aginst_Q(matrices_list, pi4, t_range)

## Correlation assessment

In [ ]:
from scipy.stats import spearmanr

grouped_low = df_low.groupby("Time")
grouped_high = df_high.groupby("Time")

Spearman-r correlation

In [ ]:
# Dictionary to store the correlation results
correlation_results_low_nabla = {}


# Loop over each group and calculate the Pearson correlation
for time, group in grouped_low:
    corr, p = spearmanr(group["Nabla_abs_diff"], group["ENS_abs_difference"])
    correlation_results_low_nabla[time] = (corr, p)


# Dictionary to store the correlation results
correlation_results_high_nabla = {}

# Loop over each group and calculate the Pearson correlation
for time, group in grouped_high:
    corr, p = spearmanr(group["Nabla_abs_diff"], group["ENS_abs_difference"])
    correlation_results_high_nabla[time] = (corr, p)

# Dictionary to store the correlation results
correlation_results_low_ingroup_jsd = {}

# Loop over each group and calculate the Pearson correlation
for time, group in grouped_low:
    corr, p = spearmanr(group["Ingroup_JSD"], group["ENS_abs_difference"])
    correlation_results_low_ingroup_jsd[time] = (corr, p)

# Dictionary to store the correlation results
correlation_results_high_ingroup_jsd = {}

# Loop over each group and calculate the Pearson correlation
for time, group in grouped_high:
    corr, p = spearmanr(group["Ingroup_JSD"], group["ENS_abs_difference"])
    correlation_results_high_ingroup_jsd[time] = (corr, p)

# Dictionary to store the correlation results
correlation_results_low_jad = {}

# Loop over each group and calculate the Pearson correlation
for time, group in grouped_low:
    corr, p = spearmanr(group["JSD_difference"], group["ENS_abs_difference"])
    correlation_results_low_jad[time] = (corr, p)

# Dictionary to store the correlation results
correlation_results_high_jad = {}

# Loop over each group and calculate the Pearson correlation
for time, group in grouped_high:
    corr, p = spearmanr(group["JSD_difference"], group["ENS_abs_difference"])
    correlation_results_high_jad[time] = (corr, p)

Nabla diff vs ENS Difference

In [ ]:
time_nabla_abs_fig_low = plot_time_grouped_scatter_2x2(
    df_low, "Nabla_abs_diff", "ENS_abs_difference"
)
time_nabla_abs_fig_high = plot_time_grouped_scatter_2x2(
    df_high, "Nabla_abs_diff", "ENS_abs_difference"
)
# Update axis titles with enhanced fonts
time_nabla_abs_fig_low.update_xaxes(
    title_text=r"$\delta (\nabla)$",
    title_font={"size": 20, "family": "latex", "color": "black"},
    row=2,
)
time_nabla_abs_fig_low.update_yaxes(
    title_text=r"$\delta (ENS)$",
    title_font={"size": 20, "family": "latex", "color": "black"},
    col=1,
)

time_nabla_abs_fig_low.show()
# time_nabla_abs_fig_low.write_image('nabla_ens_at_time_low.pdf')

In [ ]:
# Update axis titles with enhanced fonts
time_nabla_abs_fig_high.update_xaxes(
    title_text=r"$\delta (\nabla)$",
    title_font={"size": 20, "family": "latex", "color": "black"},
    row=2,
)
time_nabla_abs_fig_high.update_yaxes(
    title_text=r"$\delta (ENS)$",
    title_font={"size": 20, "family": "latex", "color": "black"},
    col=1,
)

time_nabla_abs_fig_high.show()
# time_nabla_abs_fig_high.write_image('nabla_ens_at_time_high.pdf')

Ingroup_JSD vs ENS Difference

In [ ]:
time_ingroupjsd_fig_low = plot_time_grouped_scatter_2x2(
    df_low, "Ingroup_JSD", "ENS_abs_difference"
)
time_ingroupjsd_fig_high = plot_time_grouped_scatter_2x2(
    df_high, "Ingroup_JSD", "ENS_abs_difference"
)

# Update axis titles with enhanced fonts
time_ingroupjsd_fig_low.update_xaxes(
    title_text="In-group JSD",
    title_font={"size": 20, "family": "latex", "color": "black"},
    row=2,
)
time_ingroupjsd_fig_low.update_yaxes(
    title_text=r"$\delta (ENS)$",
    title_font={"size": 20, "family": "latex", "color": "black"},
    col=1,
)

# time_ingroupjsd_fig_low.write_image('ingroup_jsd_ens_at_time_low.pdf')
time_ingroupjsd_fig_low.show()

In [ ]:
# Update axis titles with enhanced fonts
time_ingroupjsd_fig_high.update_xaxes(
    title_text="In-group JSD",
    title_font={"size": 20, "family": "Arial", "color": "black"},
    row=2,
)
time_ingroupjsd_fig_high.update_yaxes(
    title_text=r"$\delta (ENS)$",
    title_font={"size": 20, "family": "Arial", "color": "black"},
    col=1,
)

# time_ingroupjsd_fig_high.write_image('ingroup_jsd_ens_at_time_high.pdf')
time_ingroupjsd_fig_high.show()

JAD vs ENS difference

In [ ]:
time_jsd_diff_fig_low = plot_time_grouped_scatter_2x2(
    df_low, "JSD_difference", "ENS_abs_difference"
)
time_jsd_diff_fig_high = plot_time_grouped_scatter_2x2(
    df_high, "JSD_difference", "ENS_abs_difference"
)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Data provided
x_values = [0.5, 1.0, 1.5, 2.0]
correlation_factors_low_nabla = {
    time: correlation_results_low_nabla[time][0]
    for time in correlation_results_low_nabla
}
correlation_factors_high_nabla = {
    time: correlation_results_high_nabla[time][0]
    for time in correlation_results_high_nabla
}
correlation_factors_low_jad = {
    time: correlation_results_low_jad[time][0] for time in correlation_results_low_jad
}
correlation_factors_high_jad = {
    time: correlation_results_high_jad[time][0] for time in correlation_results_high_jad
}
correlation_factors_low_ingroup_jsd = {
    time: correlation_results_low_ingroup_jsd[time][0]
    for time in correlation_results_low_ingroup_jsd
}
correlation_factors_high_ingroup_jsd = {
    time: correlation_results_high_ingroup_jsd[time][0]
    for time in correlation_results_high_ingroup_jsd
}

data = {
    "Time": x_values * 6,
    "Measure": [r"$\delta (JAD)$"] * 4
    + ["Ingroup-JSD"] * 4
    + [r"$\delta (\nabla)$"] * 4
    + [r"$\delta (JAD)$"] * 4
    + ["Ingroup-JSD"] * 4
    + [r"$\delta (\nabla)$"] * 4,
    "High/Low": ["Balanced"] * 12 + ["Imbalanced"] * 12,
    "Correlation": list(correlation_factors_low_jad.values())
    + list(correlation_factors_low_ingroup_jsd.values())
    + list(correlation_factors_low_nabla.values())
    + list(correlation_factors_high_jad.values())
    + list(correlation_factors_high_ingroup_jsd.values())
    + list(correlation_factors_high_nabla.values()),
}

# Create DataFrame
df_correlation = pd.DataFrame(data)

# Create subplots for faceting by 'Balanced' and 'Imbalanced'
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Balanced", "Imbalanced"),
    shared_yaxes=True,
    horizontal_spacing=0.01,
)

# Define color mapping for each measure
color_map = {
    r"$\delta (\nabla)$": "#6fba4f",
    "Ingroup-JSD": "#f0a3f8",
    r"$\delta (JAD)$": "#67a8cd",
}

# Add traces for each High/Low condition in each subplot
for i, condition in enumerate(df_correlation["High/Low"].unique(), start=1):
    for measure in df_correlation["Measure"].unique():
        filtered_data = df_correlation[
            (df_correlation["High/Low"] == condition)
            & (df_correlation["Measure"] == measure)
        ]
        fig.add_trace(
            go.Scatter(
                x=filtered_data["Time"],
                y=filtered_data["Correlation"],
                mode="lines+markers",
                name=f"{measure}"
                if condition == "Balanced"
                else "",  # or '' instead of None
                showlegend=True
                if condition == "Balanced"
                else False,  # hide Imbalanced legend traces
                line={"color": color_map[measure], "dash": "solid", "width": 1.5},
                marker={"symbol": "circle", "size": 7},
            ),
            row=1,
            col=i,
        )

# Update layout
fig.update_layout(
    title=None,
    legend_title="Measures",
    showlegend=True,
    template="plotly_white",
    height=600,
    width=800,
    margin={"l": 10, "r": 50, "t": 50, "b": 100},
    legend={
        "title": None,
        "font": {"size": 13},
        "orientation": "h",
        "yanchor": "top",
        "y": -0.15,
        "xanchor": "center",
        "x": 0.5,
    },
)

# Update axis titles with enhanced fonts
fig.update_xaxes(
    title_font={"size": 25, "family": "Arial", "color": "black"},
    tickfont={"size": 14},
    gridcolor="lightgrey",
    row=1,
)

fig.update_yaxes(
    title_font={"size": 25, "color": "black"},
    tickfont={"size": 14},
    gridcolor="lightgrey",
    row=1,
)

# Adjust subplot titles
fig.update_xaxes(title_text="Time", row=1)
fig.update_yaxes(title_text=r"$\hat{\rho}$", row=1, col=1)
# fig.write_image('correlation_factor_time_difference_measure_from_formula.pdf')
fig.show()

In [ ]:
# old correlation statistics
correlation_data = {
    "low_nabla": [
        0.2579135456148954,
        0.1782754633352953,
        0.10193728765928865,
        0.03535182665984866,
    ],
    # "high_nabla": [0.44375090874533624, 0.3197637163742056, 0.20452965575145146, 0.10222476164733492],
    "low_ingroup_jsd": [
        0.28758594082163663,
        0.23920729496250417,
        0.19781287571613726,
        0.1613598014569678,
    ],
    # "high_ingroup_jsd": [0.4655937219528245, 0.3993184741659232, 0.34135922306667416, 0.2903908264846763],
    "low_jad": [
        0.2988539748287563,
        0.24764781922305007,
        0.20255289579107053,
        0.16286673683071506,
    ],
    # "high_jad": [0.5471632428944311, 0.485283771174017, 0.4255250633395653, 0.3704044881707177]
}

## Valid matrices stationary distirbution closer to balanced distribution

In [ ]:
from clock_project.maths.evolutionary_rate import calculate_stationary_distribution

balanced_distance = []
imbalanced_distance = []
dist_diff_list = []
for matrix_list in matrices_list:
    for matrix in [matrix_list["ingroup1"], matrix_list["ingroup2"]]:
        std_pi = calculate_stationary_distribution(np.array(matrix))
        dist1 = jsd(std_pi, pi0)
        dist2 = jsd(std_pi, pi4)
        dist_diff = dist1 - dist2
        balanced_distance.append(dist1)
        imbalanced_distance.append(dist2)
        dist_diff_list.append(dist_diff)

In [ ]:
import plotly.express as px

fig = px.histogram(
    dist_diff_list,
    labels={"x": "Distance (JSD)", "y": "Count"},
    title=None,
    color_discrete_sequence=["#58B8D1"],
)

# Update layout for presentation
fig.update_layout(
    template="plotly_white",
    margin={"l": 50, "r": 50, "t": 50, "b": 50},  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title="<b>Count</b>",  # Explicit y-axis title
    xaxis_title=r"$JSD(\pi_{\infty}, \pi_{balanced}) - JSD(\pi_{\infty}, \pi_{imbalanced})$",  # Explicit x-axis title
    yaxis_title_font={"size": 20},  # Adjust y-axis font size
    xaxis_title_font={"size": 20},  # Adjust x-axis font size
    font={"size": 16},  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=500,  # Set figure height (optional for better control)
    showlegend=False,  # Remove the legend
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=0,
    y1=400,
    line={"color": "red", "width": 5, "dash": "dashdot"},
)
# Set transparency level and add a solid line around each bar
fig.update_traces(
    opacity=0.8,  # Set the transparency (0 = fully transparent, 1 = fully opaque)
    marker_line_color="black",  # Color of the line around each bar
    marker_line_width=1.5,  # Width of the line around each bar
)

fig.show()
# fig.write_image('/Users/gulugulu/repos/PuningAnalysis/results/figures/jsd_diff_between_balanced_imbalanced.pdf')